# Disbursement Automation (Manual Stage Control)
สมุดเล่มนี้ใช้สำหรับควบคุมการรันระบบเบิกจ่ายทีละขั้นตอน เพื่อการตรวจสอบและควบคุมด้วยมือ

### 1. Setup และโหลด Config

In [ ]:
import os
import sys
import glob
import pandas as pd
from datetime import datetime
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC

# เพิ่ม path เพื่อให้ import ไฟล์ใน folder เดียวกันได้
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from config import Config
from utils import setup_logger, setup_driver, fast_login, fill_disbursement_form

logger = setup_logger()
Config.validate()
print(f"[*] Config Loaded: Budget Year {Config.BUDGET_YEAR}")

### 2. เริ่มต้น Browser และ Login

In [ ]:
driver = setup_driver()
wait = WebDriverWait(driver, 15)
fast_login(driver, wait)
print("[+] Browser ready and logged in")

### 3. อัปเดตข้อมูลผลเบิกจ่ายจากไฟล์รายสัปดาห์ (Sync Data)
ดึงค่าจากไฟล์ `hot_adj_...` มาใส่ใน `table_main_2569.xlsx` และจดบันทึกเวลาที่ 'ตรวจสอบ'

In [ ]:
source_pattern = "hot_adj_*.xlsx"
target_file = "table_main_2569.xlsx"

source_files = glob.glob(source_pattern)
if not source_files:
    print(f"[!] ไม่พบไฟล์ Source hot_adj_*.xlsx ใน folder นี้")
else:
    source_file = max(source_files, key=os.path.getctime)
    print(f"[*] กำลังดึงข้อมูลจาก: {source_file}")
    
    # อ่านไฟล์โดยข้ามบรรทัดหัวข้อ 4 บรรทัดแรก (Header อยู่ที่บรรทัด 5 หรือ index 4)
    df_source = pd.read_excel(source_file, sheet_name="งบประมาณ 2569", header=4)
    df_target = pd.read_excel(target_file)

    # อ้างอิงคอลัมน์ด้วย Index เพื่อความแม่นยำ (เนื่องจากชื่อคอลัมน์อาจมีช่องว่างแฝง)
    # Index 6: บรรทัดรายการ (G)
    # Index 10: รหัสงบประมาณ (K)
    # Index 17: เบิกจ่ายสะสม (R)
    col_idx_g = 6
    col_idx_id = 10
    col_idx_val = 17

    # 1. กรองบรรทัดรายการที่ไม่เอาดอกจัน (*)
    df_filtered = df_source[~df_source.iloc[:, col_idx_g].astype(str).str.contains('\*')]

    # 2. รวมยอดเงินตามรหัสงบประมาณ
    # ดึงชื่อคอลัมน์จริงจาก index มาใช้ใน groupby
    id_col_name = df_source.columns[col_idx_id]
    val_col_name = df_source.columns[col_idx_val]
    
    df_summed = df_filtered.groupby(id_col_name)[val_col_name].sum().reset_index()
    mapping = dict(zip(df_summed[id_col_name].astype(str), df_summed[val_col_name]))

    # 3. อัปเดตลงตารางหลัก
    update_count = 0
    def update_row(row):
        global update_count
        gfmis = str(row['รหัส GFMIS']).strip()
        if gfmis in mapping:
            row['เบิกจ่ายสะสม'] = mapping[gfmis]
            row['ตรวจสอบ'] = datetime.now().strftime("%d/%m/%Y %H:%M")
            update_count += 1
        return row

    df_target = df_target.apply(update_row, axis=1)
    df_target.to_excel(target_file, index=False)
    print(f"[+] อัปเดตข้อมูลสำเร็จ {update_count} รายการ ลงใน {target_file} เรียบร้อยแล้ว")

### 4. ไปยังหน้าเป้าหมายโครงการ (Deep Link)
คุณสามารถระบุ URL ของโครงการที่จะกรอกได้ที่นี่

In [ ]:
print("[*] กรุณาไปที่หน้า Browser และคลิกเลือกโครงการที่ต้องการกรอกข้อมูลเบิกจ่าย")

### 5. วนลูปกรอกข้อมูลจาก Excel (Batch Update)
รัน Cell นี้เพื่อเริ่มกรอกข้อมูลจากไฟล์ `table_main_2569.xlsx` ลงหน้าเว็บโดยอัตโนมัติ

In [ ]:
df_main = pd.read_excel("table_main_2569.xlsx")

for index, row in df_main.iterrows():
    # ตรวจสอบว่ามีรหัส GFMIS และมียอดเงินที่อัปเดตมาใหม่หรือไม่
    if pd.notna(row['รหัส GFMIS']) and row['เบิกจ่ายสะสม'] > 0:
        # คุณสามารถเพิ่มเงื่อนไข 'ตรวจสอบ' เพื่อกรอกเฉพาะที่เพิ่งอัปเดตวันนี้ได้
        print(f"[*] กำลังกรอกรายการ: {row['ชื่อ / ประมาณการ']}")
        
        data_to_fill = {
            'gfmis_code': str(row['รหัส GFMIS']).strip(),
            'date': datetime.now().strftime("%d/%m/%Y"), 
            'amount_contract': 0, 
            'amount_self': row['เบิกจ่ายสะสม']
        }
        
        success = fill_disbursement_form(driver, wait, data_to_fill)
        
        if success:
            print(f"  [+] กรอกข้อมูลเรียบร้อย (GFMIS: {data_to_fill['gfmis_code']})")
        else:
            print(f"  [!] ข้ามรายการเนื่องจากหาปุ่มไม่เจอ หรือ Error")
            input("เตรียมหน้าจอให้พร้อมแล้วกด Enter เพื่อไปต่อ...")

print("--- จบการทำงาน Batch Update ---")

### 6. ปิด Browser

In [ ]:
driver.quit()
print("[*] Driver closed")